In [ ]:
import os
import json
import time
import html
import re
from tqdm import tqdm
import requests
from requests.adapters import HTTPAdapter, Retry
import random

BASE_URL = "https://noeyyaly.noe.edf.fr:8088/data-ia-factory/uc202-rex-ipn/"


def make_session(timeout=600):
    session = requests.Session()
    # Pas de retry auto côté requests -> on gère nous-mêmes
    retries = Retry(
        total=0, backoff_factor=0, status_forcelist=[], allowed_methods=None
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.timeout = timeout
    return session


session = make_session(timeout=600)


def _clean_error_text(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", "", text)  # strip balises HTML
    return text.strip()


def _post_answer_with_retries(url, payload, max_attempts=3, base_delay=0.8, timeout=600):
    """
    Retry SEULEMENT pour /answer_generation :
    - Retente si:
        * Exception réseau (timeouts, connexion)
        * HTTP status >= 400
        * Body JSON == {"error": "..."}
    - NE RETENTE PAS si 2xx avec du texte quelconque (JSON ou pas).
    """
    last_err = None
    delay = base_delay

    for attempt in range(1, max_attempts + 1):
        try:
            resp = session.post(url, json=payload, timeout=timeout)
            text = (resp.text or "").strip()

            # HTTP error -> retry
            if resp.status_code >= 400:
                last_err = requests.HTTPError(f"HTTP {resp.status_code}: {text[:300]}")
                if attempt < max_attempts:
                    print(
                        f"[WARN] answer_gen attempt {attempt}/{max_attempts} failed: {last_err}. Retry in {delay:.2f}s."
                    )
                    time.sleep(delay + random.uniform(0, delay / 4))
                    delay *= 1.6
                    continue
                else:
                    return resp  # on renvoie quand même la dernière réponse pour log

            # Check body {"error": "..."} -> retry
            try:
                maybe_json = resp.json()
                if (
                    isinstance(maybe_json, dict)
                    and "error" in maybe_json
                    and len(maybe_json) == 1
                ):
                    last_err = RuntimeError(
                        _clean_error_text(maybe_json.get("error", ""))
                    )
                    if attempt < max_attempts:
                        print(
                            f"[WARN] answer_gen attempt {attempt}/{max_attempts} failed: {last_err}. Retry in {delay:.2f}s."
                        )
                        time.sleep(delay + random.uniform(0, delay / 4))
                        delay *= 1.6
                        continue
                    else:
                        return resp  # on garde la dernière réponse, même si c'est un error JSON
            except ValueError:
                # Non-JSON -> OK pour nous (on accepte tel quel)
                pass

            # Succès (quel que soit le contenu)
            return resp

        except requests.exceptions.RequestException as e:
            last_err = e
            if attempt < max_attempts:
                print(
                    f"[WARN] answer_gen attempt {attempt}/{max_attempts} failed: {e}. Retry in {delay:.2f}s."
                )
                time.sleep(delay + random.uniform(0, delay / 4))
                delay *= 1.6
            else:
                # Dernier échec: on remonte une pseudo-réponse synthétique
                class DummyResp:
                    status_code = 599
                    text = json.dumps({"error": f"{type(e).__name__}: {str(e)}"})

                    def json(self_inner):
                        try:
                            return json.loads(self_inner.text)
                        except Exception:
                            return {"error": str(e)}

                return DummyResp()


def process_question(question_text, timeout=600, model=None):
    """
    /hybrid : 1 appel simple (pas de retry).
    /answer_generation : retry sur erreurs (max 2 retries -> 3 tentatives totales).
    Sortie : question, chunks (tels que renvoyés), réponse brute du LLM (string), et info erreur.
    """
    # Valeurs par défaut si un crash survient tôt
    chunks = []
    raw_llm_response = ""
    error_msg = ""

    try:
        # Step 1: Retrieve top chunks (NO retry)
        hybrid_payload = {"query": question_text, "top_k": 100, "use_dictionary": False,"filters": {"source": ["cameleon","uc202-rex-gsimon"]}}
        resp_hybrid = requests.post(
            BASE_URL + r"hybrid_search/hybrid",
            json=hybrid_payload,
            verify="autorite_chain.pem",
        )

        try:
            resp_hybrid.raise_for_status()
            chunks = resp_hybrid.json() if resp_hybrid.ok else []
        except requests.HTTPError as he:
            # On log l'erreur mais on continue (chunks restera vide)
            error_msg = _clean_error_text(f"Hybrid HTTPError: {str(he)}")

        # Step 2: LLM answer (retry sur erreurs)
        llm_payload = {"query": question_text, "chunks": chunks}
        if model:
            llm_payload["model"] = model

        resp_llm = requests.post(
            BASE_URL + r"answer_generation/answer_generation",
            json=llm_payload,
            verify="autorite_chain.pem",
            timeout=600,
        )

        raw_llm_response = (resp_llm.text or "").strip()

        # Si c'est un JSON {"error": "..."} ou un HTTP error, on remonte aussi le message dans error_msg
        try:
            maybe_json = json.loads(raw_llm_response)
            if (
                isinstance(maybe_json, dict)
                and "error" in maybe_json
                and len(maybe_json) == 1
            ):
                error_msg = _clean_error_text(maybe_json.get("error", ""))
        except Exception:
            # Non-JSON -> pas grave
            pass

        if resp_llm.status_code >= 400 and not error_msg:
            error_msg = f"HTTP {resp_llm.status_code}"

        return {
            "question": question_text,
            "chunks": chunks,
            "llm_response": raw_llm_response,
            "error": error_msg,
        }

    except Exception as e:
        # On renvoie quand même ce que l'on a
        return {
            "question": question_text,
            "chunks": chunks,
            "llm_response": raw_llm_response,
            "error": _clean_error_text(f"Unexpected error: {str(e)}"),
        }


def safe_process_question(question_text, **kwargs):
    """Encapsule pour ne jamais lever d'exception vers la boucle."""
    try:
        return process_question(question_text, **kwargs)
    except Exception as e:
        return {
            "question": question_text,
            "chunks": [],
            "llm_response": "",
            "error": _clean_error_text(f"[fatal] {str(e)}"),
        }


def append_jsonl(result, path="annotator_report_atelier_10-12.jsonl"):
    """Écrit chaque résultat sur une ligne JSON au fur et à mesure (anti-crash)."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


def build_json_report(results, output_path="annotator_report_atelier_10-12.json"):
    """
    Consolide tout en un JSON:
    [
      {
        "question_id": "Q1",
        "question": "...",
        "chunks": [...],
        "llm_response": "texte brut (JSON ou non)",
        "error": "message d'erreur optionnel"
      },
      ...
    ]
    """
    data = []
    for i, res in enumerate(results, start=1):
        data.append(
            {
                "question_id": f"Q{i}",
                "question": res.get("question"),
                "chunks": res.get("chunks", []),
                "llm_response": res.get("llm_response", ""),
                "error": res.get("error", ""),
            }
        )
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON report saved to {os.path.abspath(output_path)}")


# -------------------------------
# Boucle principale
# -------------------------------


questions = [
    "Quel est le REX sur l’organisation des essais à chaud sur l’EPR de FA3 ?",
    "Quel est le REX sur l’organisation des essais à froid sur l’EPR de FA3 ?",
    "Quel est le REX sur l’organisation des DEM sur l’EPR",
    "Quel est le REX sur l’organisation des DEM sur l’EPR de FA3 ?",
    "Quel est le REX sur l’organisation des différentes phases d’essais de démarrage sur l’EPR de FA3 ?",
    "Quel est le REX des essais de mise en service des groupes diesels par le titulaire ?"
]


results = []
jsonl_path = "annotator_report_epria_boosted.json"

# Optionnel : purge du JSONL
if os.path.exists(jsonl_path):
    os.remove(jsonl_path)

for q in tqdm(questions, desc="Processing questions", leave=True):
    res = safe_process_question(q)
    results.append(res)
    append_jsonl(res, path=jsonl_path)
    if res.get("error"):
        print(f"[ERROR] {q}\n        {res['error']}")

# Rapport final JSON (liste complète)
build_json_report(results, output_path="annotator_report_epria_boosted.json")
print(f"📄 JSONL stream saved to {os.path.abspath(jsonl_path)}")


Processing questions:  17%|█▋        | 1/6 [03:58<19:51, 238.25s/it]

In [2]:
import json
import os


def analyze_llm_responses(
    report_path="annotator_report_epria_boosted.json", max_error_samples=10
):
    """
    Ouvre le rapport consolidé et essaie json.loads() sur chaque champ 'llm_response'.
    Retourne un dictionnaire avec les stats et quelques exemples d'échecs.
    """
    result = {
        "path": os.path.abspath(report_path),
        "exists": os.path.exists(report_path),
        "total_records": 0,
        "json_parseable_count": 0,
        "json_unparseable_count": 0,
        "parseable_indices": [],
        "unparseable_indices": [],
        "errors_sample": [],
    }

    if not result["exists"]:
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result

    try:
        with open(report_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            data = [data]

        result["total_records"] = len(data)

        for idx, rec in enumerate(data, start=1):
            llm_resp = rec.get("llm_response", "")
            try:
                json.loads(llm_resp)
                result["json_parseable_count"] += 1
                result["parseable_indices"].append(idx)
            except Exception as e:
                result["json_unparseable_count"] += 1
                result["unparseable_indices"].append(idx)
                if len(result["errors_sample"]) < max_error_samples:
                    result["errors_sample"].append(
                        {
                            "index": idx,
                            "error": str(e),
                            "snippet": (llm_resp or "")[:200],
                        }
                    )

        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result

    except Exception as e:
        result["errors_sample"].append({"fatal": str(e)})
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result


# Utilisation :
analyze_llm_responses("annotator_report_epria_boosted.json")


{
  "path": "/opt/app-root/src/uc202-ipn-rex/notebooks/annotator_report_epria_boosted.json",
  "exists": true,
  "total_records": 6,
  "json_parseable_count": 5,
  "json_unparseable_count": 1,
  "parseable_indices": [
    1,
    2,
    3,
    5,
    6
  ],
  "unparseable_indices": [
    4
  ],
  "errors_sample": [
    {
      "index": 4,
      "error": "Expecting value: line 1 column 1 (char 0)",
      "snippet": ""
    }
  ]
}


{'path': '/opt/app-root/src/uc202-ipn-rex/notebooks/annotator_report_epria_boosted.json',
 'exists': True,
 'total_records': 6,
 'json_parseable_count': 5,
 'json_unparseable_count': 1,
 'parseable_indices': [1, 2, 3, 5, 6],
 'unparseable_indices': [4],
 'errors_sample': [{'index': 4,
   'error': 'Expecting value: line 1 column 1 (char 0)',
   'snippet': ''}]}

In [3]:

import json
import os
import shutil

REPORT_PATH = (
    "/opt/app-root/src/uc202-ipn-rex/notebooks/annotator_report_epria_boosted.json"
)
BACKUP_PATH = REPORT_PATH + ".bak"


def add_llm_response_json(report_path=REPORT_PATH, backup_path=BACKUP_PATH):
    # 1) Vérifications basiques
    if not os.path.exists(report_path):
        raise FileNotFoundError(f"Fichier introuvable: {report_path}")

    # 2) Sauvegarde de sécurité
    shutil.copy2(report_path, backup_path)
    print(f"💾 Backup created: {backup_path}")

    # 3) Lecture
    with open(report_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        # Le rapport doit être une liste de dicts
        data = [data]

    total = len(data)
    parseable = 0
    unparseable = 0

    # 4) Ajout de la clé llm_response_json
    for i, rec in enumerate(data, start=1):
        raw = rec.get("llm_response", "")
        parsed = None
        # Si 'llm_response' est déjà un dict/list (rare), on le met directement
        if isinstance(raw, (dict, list)):
            parsed = raw
        else:
            try:
                parsed = json.loads(raw)
                parseable += 1
            except Exception:
                parsed = None
                unparseable += 1

        rec["llm_response_json"] = parsed

    # 5) Écriture
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"✅ Updated report saved to {os.path.abspath(report_path)}")
    print(f"📊 Stats: total={total}, parseable={parseable}, unparseable={unparseable}")


if __name__ == "__main__":
    add_llm_response_json()


💾 Backup created: /opt/app-root/src/uc202-ipn-rex/notebooks/annotator_report_epria_boosted.json.bak
✅ Updated report saved to /opt/app-root/src/uc202-ipn-rex/notebooks/annotator_report_epria_boosted.json
📊 Stats: total=6, parseable=5, unparseable=1


In [1]:
from openai import OpenAI
def get_models_list_in_portail_IAG() -> list[str]:

    api = OpenAI(
        base_url="https://oneapi.edf.fr/dteo/it/It_EdfPortailMultiIAG_OpenAI_Bearer/1.0/v2/workspaces/HcA-puQ/webhooks/v1",
        api_key="a285d9458af94721925ed04494ce1db0",
    )

    models = api.models.list()

    models_available = [model.id for model in models.data]

    return models_available


In [2]:
get_models_list_in_portail_IAG()


['C2-Interne-Mistral-Small',
 'C2-Interne-Mistral-Small-3.2',
 'C2-Interne-Mixtral-8x7b',
 'C2-Interne-Croissant',
 'C2-Interne-Mistral-7b',
 'C2-Interne-Codestral-2501',
 'C2-Interne-Codestral-2508',
 'C1-Cloud-Gemini-2.0-Flash',
 'C2-Cloud-Gemini-Embedding-001',
 'C2-Cloud-Gemini-2.5-Flash',
 'C2-Cloud-Gemini-2.5-Pro',
 'C2-Cloud-Gemini-2.0-Flash',
 'C2-Cloud-Gemini-2.5-Flash-lite',
 'bge-m3-custom-fr',
 'text-embedding-ada-002',
 'kalm-mini-it-v15',
 'modernbert-embed-base',
 'C2-Cloud-Mistral-Medium',
 'C2-Interne-Mistral-Medium-3.1',
 'C2-Cloud-Codestral-2501']

In [10]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.api.utils.portail_iag import PortailIAG

iag_portail = PortailIAG()
import json

import requests
import json

# --- ⚙️ CONFIGURATION ---
BASE_URL = "http://localhost:5001"  # ou ton adresse backend réelle (FastAPI)

# --- 🔍 Étape 1 : Simuler la requête /hybrid ---
query = "Je dois rédiger un cahier des charges pour la fourniture et le montage de réchauffeurs d’eau alimentaire. Quel est le REX d’affaires précédentes (approvisionnement et montage) à prendre en compte pour la rédaction de mon cahier des charges ? "

hybrid_payload = {"query": query, "top_k": 100, "use_dictionary": False}

resp_hybrid = requests.post(f"{BASE_URL}/hybrid", json=hybrid_payload)

print("✅ /hybrid status:", resp_hybrid.status_code)

if not resp_hybrid.ok:
    print("Erreur hybrid:", resp_hybrid.text)
else:
    chunks = resp_hybrid.json()
    print(f"Nombre de chunks reçus : {len(chunks)}")

# --- 💬 Étape 2 : Appeler /answer_generation ---
chunks_original=chunks
chunks = [
    {
        "chunk_id": d["chunk_id"],
        "chunk_content": d["chunk_content"],
    }
    for d in chunks['chunks']
]
allowed_ids = [d["chunk_id"] for d in chunks]

REFERENCE_KEY="references"

# Convertir le JSON en chaîne de caractères
#response_format_str = json.dumps(response_format, indent=4, ensure_ascii=False)


prompt = f"""Tu es un assistant expert, ingénieur spécialisé dans l’exploitation du REX ingénierie et construction EDF.
TA MISSION :
Répondre de manière complète, précise et techniquement fiable à la question suivante : {query}
TON OBJECTIF :
Produire une synthèse structurée, claire et exhaustive basée EXCLUSIVEMENT sur les extraits fournis (chunks).
RÈGLES STRICTES :
1. Couvre TOUTES les informations importantes présentes dans les chunks pertinents (chiffres, exemples, contraintes, etc.). Aucune omission.
2. N’utilise AUCUNE connaissance externe.
3. Structure la réponse en plusieurs "thèmes", chacun pouvant contenir des "sous-thèmes".
4. Pour chaque sous-thème :
   - Rédige un contenu clair, précis et détaillé.
   - Intègre tous les éléments pertinents extraits des chunks.
   - Liste les références exactes (chunk_id) utilisées pour ce sous-thème dans un champ "{REFERENCE_KEY}" de type liste.
   - **IMPORTANT** : Les valeurs de "{REFERENCE_KEY}" doivent être choisies **UNIQUEMENT** parmi la le.
5. Si un aspect de la question n’a pas d’information pertinente dans les chunks, dis-le explicitement dans le contenu du sous-thème.
7. La langue de réponse doit être la même que la question: {query}

LISTE BLANCHE DES chunk_id AUTORISÉS :
{allowed_ids} . Tous les élements de "{REFERENCE_KEY}" dans le JSON de sortie doivent provenir de cette liste.

QUESTION :
{query}
EXTRAITS :
{chunks}
Maintenant, produis une réponse exhaustive, structurée et fidèle.

Ayant produis une réponse, Tu es maintenant un convertisseur. Ta SEULE tâche est de transformer cette réponse en JSON STRICT.

Règles impératives :
- Respecte EXACTEMENT le schéma suivant :
FORMAT DE SORTIE (JSON STRICT UNIQUEMENT, SANS TEXTE AVANT/APRÈS) :
[
  {{
    "theme": "…",
    "subthemes": [
      {{
        "subtheme": "…",
        "content": "…",
        "références": ["chunk_id1", "chunk_id2"]
      }}
    ]
  }}
]
- Garantis que "références" est TOUJOURS une liste JSON (même avec un seul élément).Il ne faut pas que les références restent mentionnés dans le corps du texte. Ils doivent être mentionnés séparément dans une liste à la fin de chaque sous-thème.

- AUCUN TEXTE avant/après le JSON. Pas de ``` ni d'explications.
- Si tu n'arrives pas à mapper proprement, renvoie [] (une liste vide).
Ta réponse va être introduite dans un json.loads sur python. Assure toi que cela ne produit aucune erreur.
"""



# prompt=f"réponds à la question {query}"
MODEL_IAG = "C2-Interne-Mistral-Medium-3.1"
#generated_answer = iag_portail.query_mistral(model=MODEL_IAG, prompt=prompt, query=query)



✅ /hybrid status: 200
Nombre de chunks reçus : 2


In [12]:
        t=iag_portail.client.chat.completions.create(
            model="C2-Interne-Mistral-Medium-3.1",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": query},
            ],
            timeout=600.0,
            temperature=0.0,
            max_completion_tokens=32000,
            stream=True,
            
            
            
        )

INFO:httpx:HTTP Request: POST https://oneapi.edf.fr/dteo/it/It_EdfPortailMultiIAG_OpenAI_Bearer/1.0/v2/workspaces/HcA-puQ/webhooks/v1/chat/completions "HTTP/1.1 408 Request Timeout"
INFO:openai._base_client:Retrying request to /chat/completions in 0.423310 seconds
INFO:httpx:HTTP Request: POST https://oneapi.edf.fr/dteo/it/It_EdfPortailMultiIAG_OpenAI_Bearer/1.0/v2/workspaces/HcA-puQ/webhooks/v1/chat/completions "HTTP/1.1 500 Internal Server Error"
INFO:openai._base_client:Retrying request to /chat/completions in 0.960862 seconds
INFO:httpx:HTTP Request: POST https://oneapi.edf.fr/dteo/it/It_EdfPortailMultiIAG_OpenAI_Bearer/1.0/v2/workspaces/HcA-puQ/webhooks/v1/chat/completions "HTTP/1.1 200 OK"


In [13]:
events=list(t)
full_text=''
for event in events:
        
            delta = event.choices[0].delta
            if delta and delta.content:
                full_text += delta.content


In [14]:
from src.api.utils.query import (
    try_parse_json_strict,
    try_parse_json_relaxed,
    normalize_llm_output,
    repair_gemini_json,
    remap_unknown_references,
    count_tokens,
)

from src.api.constants.openai import (
    DEFAULT_MODEL,
    MODEL_GEMINI,
    MODEL_MISTRAL,
    MISTRAL_MODELS,
    MISTRAL_TIMEOUT_SECONDS,  # <-- timeout par défaut (secondes)
    build_prompt_for_model,
    build_mistral_jsonify_prompt,
    REFERENCE_KEY,
)


content = full_text.strip()
parsed = (
    repair_gemini_json(content)
    or try_parse_json_strict(content)
    or try_parse_json_relaxed(content)
)
parsed


[{'theme': 'Exigences techniques et réglementaires',
  'subthemes': [{'subtheme': 'Dimensionnement et spécifications techniques',
    'content': "Il est crucial de préciser dans le cahier des charges les règles de dimensionnement (thermo-hydraulique, mécanique, tenue au séisme) et les éléments de design spécifiques. Les spécifications doivent inclure la prise en compte de l'encrassement pour le dimensionnement thermique, l'intégration de la tenue au séisme, ainsi que les efforts et moments apportés par les tuyauteries. Il est également recommandé de sensibiliser les industriels en charge de la fabrication sur la culture qualité et la conservation avant raccordement sur site. Les réchauffeurs doivent être dimensionnés pour résister aux conditions de fonctionnement, y compris les températures maximales et minimales admissibles. Par exemple, pour les réchauffeurs d'eau alimentaire, il est nécessaire de vérifier les approvisionnements des presses étoupes des entrées câbles d’alimentation e

In [8]:
from src.api.utils.query import (
    try_parse_json_strict,
    try_parse_json_relaxed,
    normalize_llm_output,
    repair_gemini_json,
    remap_unknown_references,
    count_tokens,
)

from src.api.constants.openai import (
    DEFAULT_MODEL,
    MODEL_GEMINI,
    MODEL_MISTRAL,
    MISTRAL_MODELS,
    MISTRAL_TIMEOUT_SECONDS,  # <-- timeout par défaut (secondes)
    build_prompt_for_model,
    build_mistral_jsonify_prompt,
    REFERENCE_KEY,
)




content = full_text.strip()
parsed = (repair_gemini_json(content) or
    try_parse_json_strict(content)
    or try_parse_json_relaxed(content)
    
)


Secrets chargés avec succès !


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: /opt/app-root/src/uc202-ipn-rex/src/models/models/test


In [23]:
parsed = normalize_llm_output(parsed, REFERENCE_KEY)


In [31]:
prompt = f"""
Tu es un **convertisseur et normaliseur de contenu**.

🎯 OBJECTIF UNIQUE :
Transformer le texte fourni en un **JSON STRICT** respectant EXACTEMENT le schéma ci-dessous,
en **améliorant la lisibilité du contenu** (retours à la ligne, listes claires, texte fluide).

────────────────────
RÈGLES ABSOLUES (NON NÉGOCIABLES)
────────────────────
1. La sortie DOIT être un JSON strictement valide (json.loads en Python).
2. AUCUN texte avant ou après le JSON.
3. Aucun bloc ``` ou commentaire.
4. Si le mapping est impossible ou ambigu → renvoie [] (liste vide).
5. Respecte EXACTEMENT ce format :

[
  {{
    "theme": "…",
    "subthemes": [
      {{
        "subtheme": "…",
        "content": "…",
        "references": ["chunk_id1", "chunk_id2"]
      }}
    ]
  }}
]

────────────────────
RÈGLES DE NORMALISATION DU CONTENU
────────────────────
- Le champ "content" DOIT être un texte lisible et structuré.
- Introduis DES RETOURS À LA LIGNE (`\\n`) quand c’est pertinent.
- Convertis les énumérations inline (ex: "- **Point** : explication")
  en listes lisibles, par exemple :

  Exemple attendu dans "content" :
  "Les éléments suivants doivent être pris en compte :\\n
  - Encrassement : ...\\n
  - Tenue au séisme : ...\\n
  - Normes applicables : ..."

- Supprime TOUTE référence (chunk_id, code, identifiant) présente dans le texte.
- Les références doivent apparaître UNIQUEMENT dans le champ "references".
- Le champ "references" est TOUJOURS une liste JSON, même avec un seul élément.
- Ta réponse va être introduite dans un json.loads sur python. Assure toi que cela ne produit aucune erreur.

────────────────────
INTERDICTIONS
────────────────────
❌ Ne modifie pas les thèmes ou sous-thèmes.
❌ N’ajoute pas d’information.
❌ Ne résume pas le contenu.
❌ Ne conserve pas de listes compactes sur une seule ligne.
❌ Ne mélange jamais texte et références.

────────────────────
TEXTE À CONVERTIR :
{parsed}
"""

MODEL_IAG = "C2-Interne-Mistral-Small-3.2"
t=iag_portail.client.chat.completions.create(
            model=MODEL_IAG,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": ""},
            ],
            timeout=600.0,
            temperature=0.0,
            max_completion_tokens=32000,
            stream=True,
            
        )


INFO:httpx:HTTP Request: POST https://oneapi.edf.fr/dteo/it/It_EdfPortailMultiIAG_OpenAI_Bearer/1.0/v2/workspaces/HcA-puQ/webhooks/v1/chat/completions "HTTP/1.1 200 OK"


In [32]:
events = list(t)
full_text = ""
for event in events:
    delta = event.choices[0].delta
    if delta and delta.content:
        full_text += delta.content


In [36]:
full_text

'```json\n[\n  {\n    "theme": "Identification et Préparation en Amont",\n    "subthemes": [\n      {\n        "subtheme": "Repérage et Diagnostic Amiante",\n        "content": "Avant toute intervention sur des équipements ou matériaux potentiellement amiantés, un repérage et diagnostic amiante est obligatoire. Ces diagnostics doivent être intégrés aux dossiers de modification et pris en compte dès la phase d\'analyse.\\n\\nActions en cas de découverte non anticipée :\\n- Déshabillage des intervenants et mise en déchets amiantés des tenues.\\n- Balisage de la zone.\\n- Campagne de prélèvements ambiants pour évaluer la contamination.\\n\\nExemple : Lors d\'une modification sur des équipements avec joints amiantés (non pris en compte initialement), une exposition accidentelle a eu lieu. Les analyses ont confirmé la présence d\'amiante (valeur < 5 fibres/L).\\n\\nRecommandations :\\n- Réaliser un état des lieux sur les autres modifications concernées.\\n- Partager le retour d\'expérience 

In [ ]:
import json
import re
from typing import Any, List, Dict


def normalize_llm_json_output(raw_text: str) -> List[Dict[str, Any]]:
    """
    Nettoie et normalise une sortie LLM en JSON strict.
    Ne conserve que : theme, subthemes, subtheme, content, references.
    Retourne [] en cas d'erreur ou de structure invalide.
    """

    try:
        # 1. Suppression des balises ```json ``` ou ``` ```
        cleaned = re.sub(r"```(?:json)?", "", raw_text, flags=re.IGNORECASE).strip()

        # 2. Parsing JSON
        data = json.loads(cleaned)

        # 3. Validation de la structure racine
        if not isinstance(data, list):
            return []

        normalized_output = []

        for theme_block in data:
            if not isinstance(theme_block, dict):
                continue

            theme = theme_block.get("theme")
            subthemes = theme_block.get("subthemes")

            if not isinstance(theme, str) or not isinstance(subthemes, list):
                continue

            clean_subthemes = []

            for st in subthemes:
                if not isinstance(st, dict):
                    continue

                subtheme = st.get("subtheme")
                content = st.get("content")
                references = st.get("references", [])

                # Sécurité sur types
                if not isinstance(subtheme, str) or not isinstance(content, str):
                    continue

                if not isinstance(references, list):
                    references = []

                # Filtrage des références non string
                references = [ref for ref in references if isinstance(ref, str)]

                clean_subthemes.append(
                    {
                        "subtheme": subtheme.strip(),
                        "content": content.strip(),
                        "references": references,
                    }
                )

            if clean_subthemes:
                normalized_output.append(
                    {"theme": theme.strip(), "subthemes": clean_subthemes}
                )

        return normalized_output

    except Exception:
        # En cas de JSON invalide ou d'erreur imprévue
        return []



In [39]:
parsed

"Identification et Préparation en Amont\nSélectionnez un ou plusieurs sous‑thèmes ci‑dessous, puis cliquez sur « Approfondir ces thèmes ».\n\nRepérage et Diagnostic Amiante\nAvant toute intervention sur des équipements ou matériaux potentiellement amiantés, un repérage et diagnostic amiante est obligatoire. Ces diagnostics doivent être intégrés aux dossiers de modification et pris en compte dès la phase d'analyse. **Actions en cas de découverte non anticipée** : - Déshabillage des intervenants et mise en déchets amiantés des tenues. - Balisage de la zone. - Campagne de prélèvements ambiants pour évaluer la contamination. **Exemple** : Lors d'une modification sur des équipements avec joints amiantés (non pris en compte initialement), une exposition accidentelle a eu lieu. Les analyses ont confirmé la présence d'amiante (valeur < 5 fibres/L). **Recommandations** : - Réaliser un état des lieux sur les autres modifications concernées. - Partager le retour d'expérience avec les centres d'in

In [38]:
normalized = normalize_llm_json_output(full_text)

# Vérification immédiate
normalized

[{'theme': 'Identification et Préparation en Amont',
  'subthemes': [{'subtheme': 'Repérage et Diagnostic Amiante',
    'content': "Avant toute intervention sur des équipements ou matériaux potentiellement amiantés, un repérage et diagnostic amiante est obligatoire. Ces diagnostics doivent être intégrés aux dossiers de modification et pris en compte dès la phase d'analyse.\n\nActions en cas de découverte non anticipée :\n- Déshabillage des intervenants et mise en déchets amiantés des tenues.\n- Balisage de la zone.\n- Campagne de prélèvements ambiants pour évaluer la contamination.\n\nExemple : Lors d'une modification sur des équipements avec joints amiantés (non pris en compte initialement), une exposition accidentelle a eu lieu. Les analyses ont confirmé la présence d'amiante (valeur < 5 fibres/L).\n\nRecommandations :\n- Réaliser un état des lieux sur les autres modifications concernées.\n- Partager le retour d'expérience avec les centres d'ingénierie pour améliorer la prise en comp